# Practice Session 02: Data cleaning

After ensuring the dataset has no null values, we would like to do some analysis, which will require some further data cleaning.

Remember that **data cleaning** is a process by which data records are modified or deleted until each record passes data validity criteria.

The analysis will seek to determine if properties in some specific neighbhoods are overpriced or underprices, relative to similar properties of the same size. As proxies for size, we know how many bedrooms, beds, and bathrooms those properties have, and how many of those bathrooms are private or shared.

However, important numerical data is sometimes encoded as text. Moreover, there are some inconsistencies in the data. We will address these.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

⭐ **Regarding the usage of generative AI.** You may use generative AI to (a) understand how to do certain operations, and (b) generate short snippets of code, that you can incorporate into your practice after reviewing them. In that case, add a comment `#AI` before the snippet. You can also add `#AI (edited)` if you edited the code before incorporating it.

⚠️ It is forbidden to use generative AI in these practices for answering the text-based questions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Author: <font color="#2680EB">Your name here</font>

E-mail: <font color="#2680EB">Your e-mail here</font>

AI used: <font color="#2680EB">Maker and model, or "none".</font>

Date: <font color="#2680EB">The current date here</font>

In [ ]:
# LEAVE AS-IS

# Required libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 0. The dataset

Let us load `listings_clean.csv.gz`, which is **a pre-processed version**. Use the one provided in the ``data/`` directory of the course's GitHub.

Remember that the original dataset was scrapped from the Airbnb site and can be found at [insideairbnb.com](https://insideairbnb.com/get-the-data), and that a [data dictionary](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?gid=1322284596#gid=1322284596) is available.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

input_dataset = pd.read_csv("listings_clean.csv.gz", compression="gzip")

# 1. Clean (data about) bathrooms <font color="#2680EB">[20%]</font>

We would like to analyze the data regarding number of bathrooms. This data seems to be found, somewhat redundantly and somewhat inconsistently, in two different columns: `bathrooms` and `bathrooms_text`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

def bathroom_crosstab(df):
    crosstab = pd.crosstab(
        df["bathrooms"].fillna("(null)"),
        df["bathrooms_text"].fillna("(null)")
    )
    display(crosstab)
    
bathroom_crosstab(input_dataset)

We notice that there is some information on whether bathrooms are **private** or **shared** that is relevant, but is encoded in a text field. First, let us ensure that text field (`bathrooms_text`) is valid according to a constraint. These are the only valid values:

* "0 baths"
* "1 bath"
* "1 private bath"
* "1 shared bath"
* "N baths"
* "N shared baths"

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS
def check_bathrooms_text(df):
    
    # Extract number and text
    # The instructor used AI for the regexp (Claude Sonnet 5)
    extracted = df["bathrooms_text"].str.extract(r"^(\d+(?:\.\d+)?)\s(.*)$")
    extracted_number = extracted[0].astype(float)
    extracted_text = extracted[1]

    # Determine if singular or plural
    is_singular = extracted_text.isin(["bath", "private bath", "shared bath"])
    is_plural = extracted_text.isin(["baths", "private baths", "shared baths"])
    
    # Determine if private, otherwise shared
    is_shared = extracted_text.isin(["shared bath", "shared baths"])
        
    # 
    is_correct = \
        ((extracted_number == 0) & is_plural & ~is_shared) | \
        ((extracted_number == 0.5) & is_plural) | \
        ((extracted_number == 1) & is_singular) | \
        ((extracted_number > 1) & is_plural) 

    invalid_rows = df[~is_correct]
    if len(invalid_rows) > 0:
        print(f"Rows that do not pass the check: {len(invalid_rows)}")
        display(invalid_rows["bathrooms_text"].value_counts().reset_index())
        return False
    else:
        return True

check_bathrooms_text(input_dataset)
    

<font size="+1" color="#2680EB">**Implement code to fix bathrooms_text**. Make sure all rows follow the constraint, fix the ones that do not. Do not drop any data. Use the function above that test that your data is now valid in this regard.</font>

In [ ]:
def prepare_bathrooms_text(df):
    
    # YOUR CODE HERE
    return df

df2 = prepare_bathrooms_text(input_dataset)
check_bathrooms_text(df2)

The information on whether bathrooms are **private** or **shared** is relevant, but is encoded in a text field. Let us expand it as follows:

1. `bathrooms` should contain the number of bathrooms. If this is incompatible with the `bathrooms_text`, we drop the row.
2. A new column, `bathrooms_type` should include the type as `private` or `shared` according to `bathrooms_text`. If this is unspecified, we will assume it is private.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement the code to prepare data about bathrooms**. Print the number of rows that were dropped</font>

In [ ]:
def prepare_data_bathrooms(df):

    original_len = len(df)
    number_in_bathrooms_text = df["bathrooms_text"].str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
    
    # YOUR CODE HERE
    
    modified_len = len(df)
    print(f"Original: {original_len}, new: {modified_len}, dropped: {original_len - modified_len}")
    return df
    
df3 = prepare_data_bathrooms(df2)
display(df3[["bathrooms", "bathrooms_text", "bathrooms_type"]].drop_duplicates().sort_values("bathrooms"))

Now we can analyze this data, for instance by comparing histograms.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

ax = sns.histplot(data=df3, x="bathrooms", 
                  stat="probability", common_norm=False,
                  hue="bathrooms_type", multiple="dodge", shrink=3)
ax.legend_.set_title("Type of bathroom")
plt.xticks(np.arange(0, 5.5, 0.5))
_ = plt.xlim(0, 5)

<font size="+1" color="#2680EB">**Describe what you see in the histogram above**.</font>

**Answer:** According to the histogram above:

* (Your answer here.)

# 2. Fix inconsistencies in bedrooms and beds <font color="#2680EB">[20%]</font>

Let us now look at the number of bedrooms and beds.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

def bedrooms_crosstab(df):
    crosstab = pd.crosstab(
        df["bedrooms"].fillna("(null)"),
        df["beds"].fillna("(null)")
    )
    display(crosstab)
    
bedrooms_crosstab(df3)

<font size="+1" color="#2680EB">**Describe the inconsistencies that can be observed in this matrix**. Answer the following questions.</font>

**Answer:**

In terms of bedrooms and beds, what does it mean in this case that most large values can be found near the diagonal?

* (Your answer here.)

Do you see any gross inconsistency between the number of bedrooms and beds in some cases? Can you give an example of such inconsistency?

* (Your answer here.)

Assume that the number of beds is always correct, but that the number of bedrooms could have been miscounted. What validity criteria would you apply over the number of bedrooms, and what would you do in cases of violation of each criteria? For each validity criteria, indicate whether you will *fix* the data and how, or whether you will *drop* the row.

* A property with 0 beds should have 0 bedrooms: otherwise fix by setting bedrooms to 0.
* (Continue the answer here.)

We will drop invalid rows in the data, but this may entail dropping some special cases that are actually correct.  Which correct cases do you think your rules may erroneously discard?

* (Your answer here.)


To check validity of rows in a dataframe using a complex validation, you can use code like this:

* `valid1 = df["col1"] > 0`
* `valid2 = df["col2"] < 0`
* ...
* `invalid_rows = df[~(valid1 & valid2 & ...)]`

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement a check for your validy criteria**. Print the combinations of beds and bedrooms that fail the criteria.</font>

In [ ]:
def check_bedrooms_beds(df):
    
    # YOUR CODE HERE
    
    if len(invalid_rows) > 0:
        print(f"Invalid rows: {len(invalid_rows)}")
        display(invalid_rows.groupby(["bedrooms", "beds"]).size().reset_index(name="count"))
        return False
    else:
        return True

check_bedrooms_beds(df3)

<font size="+1" color="#2680EB">**Implement data cleaning to ensure the validy criteria**. You can fix data or drop rows. If you drop rows, indicate how many did you drop.</font>

In [ ]:
def fix_bedrooms_bed(df):
    df = df.copy()
    
    # YOUR CODE HERE
    return df

df4 = fix_bedrooms_bed(df3)
check_bedrooms_beds(df4)

Now we can analyze this data, for instance by creating a heatmap of `bedrooms` and `beds`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# The instructor used AI to understand crosstab filtering and Seaborn parameters (Claude Sonnet 5)
crosstab = pd.crosstab(df4["bedrooms"], df4["beds"])
crosstab = crosstab.loc[crosstab.index <= 10, crosstab.columns <= 10]
sns.heatmap(crosstab, cmap="Blues")

# 3. Categorize properties by size <font color="#2680EB">[10%]</font>

We will group properties into categories by size. We will do it by using the data that we have on beds, bedrooms, and bathrooms.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 3.1. Fix fractional bathrooms

For this analysis, we will consider that half a bathroom does not count, so we will convert 0.5 to 0, 1.5 to 1, and so on. To compute the floor of a number, one option is to use the `//` operator in Python, which divides and drop the remainder.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement code to convert the number of bathrooms**. Compute the floor of the number of bathrooms.</font>

In [ ]:
def remove_fractional_bathrooms(df):
    df = df.copy()
    # YOUR CODE HERE
    return df
   
df5 = remove_fractional_bathrooms(df4)

Let us look at a heatmap between `beds` and `bathrooms`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# The instructor used AI for crosstab filtering and Seaborn parameters (Claude Sonnet 5)
crosstab = pd.crosstab(df5["beds"], df5["bathrooms"])
crosstab = crosstab.loc[crosstab.index <= 10, crosstab.columns <= 10]
sns.heatmap(crosstab, cmap="Blues")

In the heatmap above, we can see that properties having 3 or 4 beds may have a second bathroom, and that properties having 5 beds or more are more likely to have a second bathroom than to have a single bathroom.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 3.2. Create categories by size

We will now do a categorization. One category will be properties having a single bed, we will call this category "S" (for small). Then, we will have a medium category ("M"), a large category ("L"), and an extra large category ("XL").

Pandas allows for the creation of "ordinal" data types using `CategoricalDtype`, as we use in the code below.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Add a column "category" to the properties**. This can be "S", "M", "L", or "XL", and can depend on the number of beds, bedrooms, and/or bathrooms. Make sure that each category contains at least 500 properties.</font>

In [ ]:
def add_category(df):
    df = df.copy()
    
    small = (df["beds"] == 1)
    df.loc[small, "category"] = "S"

    # YOUR CODE HERE
    
    # Convert size to ordinal variable
    size_ordinal = pd.CategoricalDtype(categories=["S", "M", "L", "XL"], ordered=True)
    df["category"] = df["category"].astype(size_ordinal)
    
    return df

df6 = add_category(df5)

display(df6.groupby(["category"], observed=True).size().reset_index(name="count"))


# 4. Analyze property size per neighborhood <font color="#2680EB">[10%]</font>

Now we will look at property sizes per area (`neibourhood_group_cleansed` in the dataset) and per neighborhood.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

crosstab = pd.crosstab(df6["neighbourhood_group_cleansed"], df6["category"])
crosstab = crosstab.loc[crosstab.sum(axis=1).sort_values().index]
# The instructor used AI for stacked plot (Claude Sonnet 5)
crosstab.plot(kind="barh", stacked=True)
plt.xlabel("count")
_ = plt.ylabel("area")

In [ ]:
# LEAVE AS-IS

crosstab = pd.crosstab(df6["neighbourhood_cleansed"], df6["category"])
crosstab = crosstab[crosstab.sum(axis=1) >= 150]  # Only neighborhoods with at least this many properties
crosstab = crosstab.loc[crosstab.sum(axis=1).sort_values().index]
crosstab.plot(kind="barh", stacked=True)
plt.xlabel("count")
_ = plt.ylabel("neighborhood")


<font size="+1" color="#2680EB">**Describe what you see in the plots above**.</font>

**Answer:**

What do you observe regarding the areas and neighborhoods that concentrate the larger number of listings?

* (Your answer here.)

What do you observe regarding the proportion of different property sizes per neighborhood?

* (Your answer here.)

# 5. Group property types <font color="#2680EB">[10%]</font>

As you can see next, there are more than 40 property types described in the column `property_type`. Add a new column `property_type_group` in which you group property types as follows:

* Rooms in an apartment, hostel, hotel, etc. = `room`
* Entire apartments, homes, lofts, condos, etc. = `unit`

Other, including campers, boats, etc., should be dropped.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# The instructor used AI (Claude Sonnet 5)
display(df6.groupby(["property_type"], observed=True)
        .size().reset_index(name="count")
        .sort_values("count", ascending=False))


<font size="+1" color="#2680EB">**Add a column "property_type_group"**. Use the specification above.</font>

In [ ]:
def add_property_type_group(df):
    df = df.copy()
    
    # YOUR CODE HERE
    return df

df7 = add_property_type_group(df6)
display(df7.groupby(["property_type", "property_type_group"], observed=True)
        .size().reset_index(name="count")
        .sort_values(["property_type_group", "count"], ascending=False))


We can now look at the distribution per area (what the original dataset calls "neighborhood groups").

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

crosstab = pd.crosstab(df7["neighbourhood_group_cleansed"],
                       df7["property_type_group"])
crosstab = crosstab.loc[crosstab.sum(axis=1).sort_values().index]
ax = crosstab.plot(kind="barh", stacked=True)
ax.legend_.set_title("Property type")
plt.xlabel("count")
_ = plt.ylabel("area")

<font size="+1" color="#2680EB">**Describe what you see in the plot above**.</font>

**Answer:**

What do you observe regarding the property types per area?

* (Your answer here.)

# 6. Convert price to a numerical value <font color="#2680EB">[10%]</font>

Convert "price" to a numerical value. It is currently expressed as a string with a dollar sign and thousands separators.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Write code to convert the column price to a numerical value**.</font>

In [ ]:
def convert_price_to_numeric(df):
    df = df.copy()
    
    # YOUR CODE HERE

    return df
    
df8 = convert_price_to_numeric(df7)

The following shows you the distribution of price (per night, in USD) per area.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

order = df8.groupby("neighbourhood_group_cleansed")["price"].mean().sort_values(ascending=False).index
sns.boxplot(data=df8, y="neighbourhood_group_cleansed", x="price", order=order)

plt.xlim(0, 800)
plt.xlabel("price per night [USD]")
plt.ylabel("neighborhood")

# 7. Analyze price and property size per area <font color="#2680EB">[20%]</font>

Next, perform an analysis of price per property size and area. For the purposes of this analysis, focus on entire units, discarding rooms.

The question we are trying to answer is: to what extent are average rental prices per neighborhood dependent on the fact that there is a larger proportion of large units in that neighborhood?

The following plot inspires this analysis: notice, for instance, that "large" units (category "L") have similar price distributions across several areas of the city. Perhaps what we see above, that some neighborhoods are more expensive than others, could be solely because they have a bigger proportion of large units.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

df9 = df8[df8["property_type_group"] == "unit"]

order = df9.groupby("neighbourhood_group_cleansed")["price"].mean().sort_values(ascending=False).index
# The instructor used AI for boxplot parameters (Claude Sonnet 5)
sns.boxplot(data=df9,
            y="neighbourhood_group_cleansed",
            x="price",
            hue="category",
            order=order)

plt.xlim(0, 800)
plt.xlabel("price per night [USD]")
plt.ylabel("neighborhood")

## 7.1. Relative price

We will first tackle this question by computing a *relative price*, that will indicate for each property, the ratio between its price and the average price of an apartment of that size.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement a computation of relative price**. First, compute the average price per size (S/M/L/XL), then, create the column *relative_price* as price/average price.</font>

In [ ]:
def add_relative_price(df):
    df = df.copy()
    
    # YOUR CODE HERE
    return df

df10 = add_relative_price(df9)
df10.groupby("neighbourhood_group_cleansed")["price_relative"].mean().reset_index().sort_values("price_relative", ascending=False)

<font size="+1" color="#2680EB">**Write your observations on this relative price result**. 
</font>

**Answer:**

What do you observe regarding relative price?

* (Your answer here.)

## 7.2. Size-adjusted average price

Next we will compute a size-adjusted average price. This is the average price in an area, if we were to replace all prices for a unit by the city-wide average price for a unit of that size.

For instance, suppose that there were two sizes A and B, and the average price for A is X, while the average price for B is Y. Suppose that in an area there are M apartments of size A and N apartments of size B. The size-adjusted average price for that area would be (M*X + N*Y)/(M+N).

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement a computation of size-adjusted price**. First, compute the average price per size (S/M/L/XL), then, compute a size-adjusted relative price. The output should be a dataframe with columns "neighbourhood_group_cleansed" and "size_adjusted_avg_price".</font>

In [ ]:
def compute_size_adjusted_price(df):

    avg_price = df.groupby("category", observed=True)["price"].mean()

    # YOUR CODE HERE

    return size_adjusted_price

size_adjusted_price = compute_size_adjusted_price(df10)
display(size_adjusted_price.reset_index().sort_values(["size_adjusted_avg_price"], ascending=False))

<font size="+1" color="#2680EB">**Write your observations on this size-adjusted average price result**.
</font>

**Answer:**

What do you observe regarding size-adjusted average price?

* (Your answer here.)

Comparing the results you obtain using the relative price, is there any robust conclusion you can reach? 

* (Your answer here.)

What are all the limitations of this analysis? Name at least three.

* (Your answer here.)


# DELIVER (individually)

Remember to read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/data-mining-course/blob/master/upf/upf-evaluation.md).

Deliver this notebook, renamed as `PSxx_Unnnnnn_YourName.ipynb` where:
* *PSxx* is the practice number.
* *Unnnnnn* is your UPF id number.
* *YourName* is your name.

## Extra points available

For more learning and extra points, let us ignore categorical sizes and instead compute a bed-adjusted price for each rental. First, compute the average price per bed across the entire city. Then, compare the actual price of a property with its price if we were charging per bed the average. 

**Note:** if you go for the extra points, add an additional section ``# <font color="#2680EB">Additional results: bed-adjusted analysis</font>`` at the end of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that I have not copied it from others, or from external sources without citing them, and that I have followed the generative AI instructions stated above.</font>